In [1]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Random Forest Baseline for Optimal Weighting Exponent $n^*$\n",
    "\n",
    "This notebook trains a machine learning model (Random Forest regressor) to predict the **optimal weighting exponent** $n^*$ for the Northstar gravitational-wave localization algorithm.\n",
    "\n",
    "It assumes you have already generated:\n",
    "- `manifest.csv` — ground-truth parameters for each event  \n",
    "- `results.csv` — performance for each $(\\text{event}, n)$ pair (not strictly needed here)  \n",
    "- `labels.csv` — best/optimal $n^*$ for each event  \n",
    "\n",
    "All three are expected to live in:\n",
    "```text\n",
    "/Users/jeem/Desktop/NorthStar_ML/data/injections\n",
    "```\n",
    "\n",
    "You can adapt paths or feature choices as needed.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "\n",
    "from sklearn.ensemble import RandomForestRegressor\n",
    "from sklearn.model_selection import train_test_split\n",
    "from sklearn.metrics import r2_score, mean_squared_error\n",
    "\n",
    "import joblib\n",
    "\n",
    "# Path to data directory (edit if you move things)\n",
    "BASE_DIR = \"/Users/jeem/Desktop/NorthStar_ML/data/injections\"\n",
    "\n",
    "manifest_path = os.path.join(BASE_DIR, \"manifest.csv\")\n",
    "labels_path   = os.path.join(BASE_DIR, \"labels.csv\")\n",
    "results_path  = os.path.join(BASE_DIR, \"results.csv\")  # optional, for later analysis\n",
    "\n",
    "print(\"Manifest path:\", manifest_path)\n",
    "print(\"Labels path:  \", labels_path)\n",
    "print(\"Results path: \", results_path)\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load core tables\n",
    "manifest = pd.read_csv(manifest_path)\n",
    "labels   = pd.read_csv(labels_path)\n",
    "\n",
    "print(\"manifest columns:\", manifest.columns.tolist())\n",
    "print(\"labels columns:  \", labels.columns.tolist())\n",
    "\n",
    "# Merge on 'event' key\n",
    "df = manifest.merge(labels, on=\"event\", suffixes=(\"\", \"_lbl\"))\n",
    "print(f\"Merged dataset shape: {df.shape}\")\n",
    "df.head()\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Quick sanity checks\n",
    "\n",
    "display(df.describe(include=\"all\").T)\n",
    "\n",
    "print(\"\\nUnique n_star values (if column name differs, edit below):\")\n",
    "# Try a few likely column names for n*\n",
    "possible_n_cols = [\"n_star\", \"n_opt\", \"best_n\", \"n_optimal\"]\n",
    "n_col = None\n",
    "for c in possible_n_cols:\n",
    "    if c in df.columns:\n",
    "        n_col = c\n",
    "        break\n",
    "\n",
    "if n_col is None:\n",
    "    raise KeyError(\"Could not find n_star column. Check labels.csv for the correct column name (e.g., 'n_star').\")\n",
    "\n",
    "print(f\"Using column '{n_col}' as target.\")\n",
    "print(df[n_col].value_counts().sort_index())\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Feature Selection\n",
    "\n",
    "We start with a simple, physically-motivated feature set:\n",
    "\n",
    "- `f` — central frequency  \n",
    "- `snr` — target SNR  \n",
    "- `theta`, `phi`, `psi` — source angles  \n",
    "- `iota` — inclination  \n",
    "\n",
    "You can later add more features such as beam-pattern coefficients or derived quantities.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Choose feature columns (edit if your column names differ)\n",
    "candidate_features = [\"f\", \"snr\", \"theta\", \"phi\", \"psi\", \"iota\"]\n",
    "\n",
    "missing = [c for c in candidate_features if c not in df.columns]\n",
    "if missing:\n",
    "    raise KeyError(f\"Missing expected feature columns: {missing}. Check manifest.csv headers.\")\n",
    "\n",
    "X = df[candidate_features].copy()\n",
    "y = df[n_col].copy()\n",
    "\n",
    "print(\"Feature matrix shape:\", X.shape)\n",
    "print(\"Target vector shape: \", y.shape)\n",
    "X.head()\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Train / Test Split and Random Forest Baseline\n",
    "\n",
    "We hold out 20% of the events as a test set and train a Random Forest regressor on the remaining 80%.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "X_train, X_test, y_train, y_test = train_test_split(\n",
    "    X, y, test_size=0.2, random_state=42\n",
    ")\n",
    "\n",
    "rf = RandomForestRegressor(\n",
    "    n_estimators=300,\n",
    "    max_depth=None,\n",
    "    min_samples_split=2,\n",
    "    random_state=42,\n",
    "    n_jobs=-1\n",
    ")\n",
    "\n",
    "rf.fit(X_train, y_train)\n",
    "\n",
    "y_pred = rf.predict(X_test)\n",
    "\n",
    "r2 = r2_score(y_test, y_pred)\n",
    "rmse = np.sqrt(mean_squared_error(y_test, y_pred))\n",
    "\n",
    "print(f\"Random Forest Baseline Performance:\")\n",
    "print(f\"  R²   = {r2:.4f}\")\n",
    "print(f\"  RMSE = {rmse:.4f}\")\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Feature Importance\n",
    "\n",
    "We inspect which input features contribute most strongly to the prediction of $n^*$.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "importances = pd.Series(rf.feature_importances_, index=candidate_features)\n",
    "importances = importances.sort_values(ascending=True)\n",
    "\n",
    "plt.figure(figsize=(6, 4))\n",
    "importances.plot(kind=\"barh\")\n",
    "plt.xlabel(\"Feature Importance\")\n",
    "plt.title(\"Random Forest Feature Importances for $n^*$ Prediction\")\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "importances\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Predicted vs True $n^*$\n",
    "\n",
    "Here we visualize how closely the model's predictions match the true labels.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "results_df = pd.DataFrame({\n",
    "    \"true_n\": y_test.values,\n",
    "    \"pred_n\": y_pred\n",
    "})\n",
    "\n",
    "plt.figure(figsize=(5, 5))\n",
    "plt.scatter(results_df[\"true_n\"], results_df[\"pred_n\"], alpha=0.3, s=10)\n",
    "min_n = min(results_df[\"true_n\"].min(), results_df[\"pred_n\"].min())\n",
    "max_n = max(results_df[\"true_n\"].max(), results_df[\"pred_n\"].max())\n",
    "plt.plot([min_n, max_n], [min_n, max_n], \"k--\", linewidth=1)\n",
    "plt.xlabel(\"True $n^*$\")\n",
    "plt.ylabel(\"Predicted $n^*$\")\n",
    "plt.title(\"True vs Predicted $n^*$\")\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "plt.figure(figsize=(6, 3))\n",
    "residuals = results_df[\"pred_n\"] - results_df[\"true_n\"]\n",
    "plt.hist(residuals, bins=30, edgecolor=\"black\", alpha=0.7)\n",
    "plt.xlabel(\"Prediction Error (pred - true)\")\n",
    "plt.ylabel(\"Count\")\n",
    "plt.title(\"Residual Distribution\")\n",
    "plt.tight_layout()\n",
    "plt.show()\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Save Trained Model\n",
    "\n",
    "We save the trained Random Forest to disk so it can be loaded later in a production or real-time pipeline.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "model_path = os.path.join(BASE_DIR, \"rf_optimal_n_model.pkl\")\n",
    "joblib.dump(rf, model_path)\n",
    "print(f\"Saved Random Forest model to: {model_path}\")\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Next Steps\n",
    "\n",
    "Some natural next directions:\n",
    "\n",
    "1. **Feature Engineering**\n",
    "   - Add additional features:\n",
    "     - Beam-pattern coefficients $F^+_H, F^\\times_H, F^+_L, F^\\times_L$\n",
    "     - Time delay $\\tau$\n",
    "     - Functions of angles (e.g., $\\sin\\theta, \\cos\\theta$) for better handling of periodicity.\n",
    "2. **Hyperparameter Tuning**\n",
    "   - Use `GridSearchCV` or `RandomizedSearchCV` to find better values of:\n",
    "     - `n_estimators`\n",
    "     - `max_depth`\n",
    "     - `min_samples_split`\n",
    "     - `max_features`\n",
    "3. **Model Comparison**\n",
    "   - Compare the Random Forest baseline to:\n",
    "     - Gradient-boosted trees (e.g., XGBoost, LightGBM, HistGradientBoostingRegressor)\n",
    "     - Small neural networks\n",
    "     - Physics-informed neural networks (PINNs) that encode weighting-function structure.\n",
    "4. **Integration with Northstar**\n",
    "   - Replace the fixed weighting exponent in Northstar with:\n",
    "     - A direct call to this trained model, or\n",
    "     - A lookup table over a discretized feature space.\n",
    "   - Measure end-to-end impact on localization metrics (e.g., $\\delta F_{\\mathrm{rms}}$, $\\delta \\tau_{\\mathrm{rms}}$) on held-out events.\n",
    "\n",
    "This notebook provides the **baseline ML result** you can now write up in your thesis:\n",
    "- Dataset size and generation procedure\n",
    "- Model architecture (Random Forest)\n",
    "- Baseline R² / RMSE\n",
    "- Dominant features\n",
    "- Example plots\n"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.x"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}


NameError: name 'null' is not defined